# MegaDescriptor 파인튜닝 (ArcFace)

wildlife-tools 레시피: ArcFace `margin=0.5`, `scale=64`, timm Swin 백본.

- 데이터: **개체당 폴더** 구조 (`<root>/<개체ID>/*.jpg`). `processed_animals`, MPDD 둘 다 이 형태로.
- split: 개체 disjoint (train 개체 / val 개체가 겹치지 않음)
- 평가: gallery/query split 의 mAP + Top-1/Top-5
- `torch+cpu` 환경에서는 사실상 못 돌린다. 아래 셀에서 CUDA 확인.


In [1]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import timm
import torchvision.transforms as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| DEVICE:", DEVICE)


torch 2.13.0+cpu | CUDA: False | DEVICE: cpu


c:\hyeonkyu\for_mate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 설정

In [2]:
CONFIG = {
    "data":         "processed_animals",              # 개체당 폴더 루트  (csv 쓰면 None)
    "csv":          None,                              # "identity,path" CSV 경로 (data 대신)
    "root":         ".",                               # csv 사용 시 path 기준 루트
    "model":        "hf-hub:BVRA/MegaDescriptor-B-224",# B-224 로 시작. L-384 는 GPU 여유 있을 때
    "epochs":       20,
    "pk_p":         12,                                # 배치 P개체
    "pk_k":         4,                                 # 개체당 K장  (배치크기 = P*K)
    "lr_head":      1e-4,
    "lr_backbone":  1e-5,
    "wd":           1e-4,
    "freeze_until": 2,     # 학습시킬 첫 Swin stage(0~3). -1=전체 학습, 99=헤드만(linear probe)
    "val_frac":     0.2,
    "seed":         0,
    "out":          "megadescriptor_ft.pt",
}
C = CONFIG
random.seed(C["seed"]); np.random.seed(C["seed"]); torch.manual_seed(C["seed"])
EXTS = (".jpg", ".jpeg", ".png", ".webp")


## 데이터: DataFrame 구성 + 개체 disjoint split

In [3]:
def df_from_folder(root):
    rows = []
    for d in sorted(os.listdir(root)):
        dd = os.path.join(root, d)
        if not os.path.isdir(dd):
            continue
        for f in sorted(os.listdir(dd)):
            if f.lower().endswith(EXTS):
                rows.append({"identity": d, "path": os.path.join(d, f)})
    return pd.DataFrame(rows)


def split_by_identity(df, val_frac=0.2, seed=0):
    rng = np.random.default_rng(seed)
    ids = df["identity"].unique().copy()
    rng.shuffle(ids)
    val_ids = set(ids[: int(len(ids) * val_frac)])
    tr = df[~df["identity"].isin(val_ids)].reset_index(drop=True)
    va = df[df["identity"].isin(val_ids)].reset_index(drop=True)
    return tr, va


if C["csv"]:
    df = pd.read_csv(C["csv"]); ROOT = C["root"]
else:
    df = df_from_folder(C["data"]); ROOT = C["data"]

tr, va = split_by_identity(df, C["val_frac"], C["seed"])

id2lab = {c: i for i, c in enumerate(sorted(tr["identity"].unique()))}
tr = tr.copy(); tr["label"] = tr["identity"].map(id2lab)
va = va.copy(); va["label"] = va["identity"].astype("category").cat.codes  # 평가용 임시 라벨
N_CLASSES = len(id2lab)

print(f"train {len(tr)}장 / {N_CLASSES}개체")
print(f"val   {len(va)}장 / {va['identity'].nunique()}개체")


train 1099장 / 276개체
val   259장 / 68개체


C:\Users\Admin\AppData\Local\Temp\ipykernel_7260\2175476699.py:16: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(ids)


## 모델 + transform

In [4]:
model = timm.create_model(C["model"], pretrained=True, num_classes=0).to(DEVICE)
cfg = timm.data.resolve_model_data_config(model)
SIZE = cfg["input_size"][-1]
print("input size:", SIZE, "| embed dim:", model.num_features, "| mean/std:", cfg["mean"], cfg["std"])

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize(cfg["mean"], cfg["std"]),
])
eval_tf = timm.data.create_transform(**cfg, is_training=False)


c:\hyeonkyu\for_mate\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--BVRA--MegaDescriptor-B-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


input size: 224 | embed dim: 1024 | mean/std: [0.485, 0.456, 0.406] [0.229, 0.224, 0.225]


## Dataset + P×K 샘플러 + ArcFace 헤드

In [5]:
class ReIDDataset(Dataset):
    def __init__(self, df, root, tf):
        self.paths = df["path"].tolist()
        self.y = df["label"].to_numpy()
        self.root, self.tf = root, tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.paths[i])).convert("RGB")
        return self.tf(img), int(self.y[i])


class PKSampler(Sampler):
    """배치 = P개체 x K장. metric learning 필수."""
    def __init__(self, labels, p, k, steps=None):
        self.labels = np.asarray(labels)
        self.p, self.k = p, k
        self.by_id = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        self.ids = list(self.by_id)
        self.steps = steps or max(1, len(self.labels) // (p * k))

    def __iter__(self):
        for _ in range(self.steps):
            picks = np.random.choice(self.ids, self.p, replace=len(self.ids) < self.p)
            batch = []
            for c in picks:
                idx = self.by_id[c]
                batch += list(np.random.choice(idx, self.k, replace=len(idx) < self.k))
            yield batch

    def __len__(self):
        return self.steps


class ArcFaceHead(nn.Module):
    def __init__(self, in_dim, n_classes, s=64.0, m=0.5):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes, in_dim))
        nn.init.xavier_uniform_(self.W)
        self.s = s
        self.cos_m, self.sin_m = np.cos(m), np.sin(m)
        self.th, self.mm = np.cos(np.pi - m), np.sin(np.pi - m) * m

    def forward(self, feats, labels):
        cos = F.linear(F.normalize(feats), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1.0 - cos ** 2)
        phi = cos * self.cos_m - sin * self.sin_m            # cos(theta + m)
        phi = torch.where(cos > self.th, phi, cos - self.mm)  # 단조성 가드
        oh = F.one_hot(labels, cos.size(1)).float()
        return (oh * phi + (1.0 - oh) * cos) * self.s


## 평가 함수 (mAP + CMC)

In [6]:
def _ap_cmc(rel):
    if not rel.any():
        return 0.0, 0, 0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())


@torch.no_grad()
def extract(model, loader):
    model.eval()
    embs, labs = [], []
    for x, y in loader:
        v = F.normalize(model(x.to(DEVICE)))
        embs.append(v.cpu().numpy()); labs.append(y.numpy())
    return np.concatenate(embs), np.concatenate(labs)


def evaluate_map_split(emb, labels, gallery_frac=0.5, seed=0):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    g_idx, q_idx = [], []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0].copy()
        rng.shuffle(idx)
        if len(idx) <= 1:
            g_idx += idx.tolist(); continue
        k = max(1, round(len(idx) * gallery_frac))
        g_idx += idx[:k].tolist(); q_idx += idx[k:].tolist()
    g_idx, q_idx = np.array(g_idx), np.array(q_idx)
    g_emb, g_lab = emb[g_idx], labels[g_idx]
    sim = emb[q_idx] @ g_emb.T
    aps = c1 = c5 = 0.0
    for row, gt in zip(sim, labels[q_idx]):
        ap, h1, h5 = _ap_cmc(g_lab[np.argsort(row)[::-1]] == gt)
        aps += ap; c1 += h1; c5 += h5
    q = len(q_idx)
    return aps / q, c1 / q, c5 / q


def evaluate_map_split_mean(emb, labels, seeds=range(5)):
    r = np.array([evaluate_map_split(emb, labels, seed=s) for s in seeds])
    return r.mean(0), r.std(0)   # ([mAP,T1,T5] 평균, 표준편차)


## 백본 freeze + 옵티마이저 + 로더

In [7]:
def set_trainable(model, freeze_until):
    for p in model.parameters():
        p.requires_grad = True
    if freeze_until < 0:
        return
    for p in model.patch_embed.parameters():
        p.requires_grad = False
    stages = list(model.layers) if hasattr(model, "layers") else []
    for i, st in enumerate(stages):
        for p in st.parameters():
            p.requires_grad = i >= freeze_until


set_trainable(model, C["freeze_until"])
head = ArcFaceHead(model.num_features, N_CLASSES).to(DEVICE)

n_train_bb = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 대상 backbone 파라미터: {n_train_bb/1e6:.1f}M / 전체 {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

tr_ld = DataLoader(
    ReIDDataset(tr, ROOT, train_tf),
    batch_sampler=PKSampler(tr["label"].to_numpy(), C["pk_p"], C["pk_k"]),
    num_workers=4, pin_memory=True,
)
va_ld = DataLoader(ReIDDataset(va, ROOT, eval_tf), batch_size=64, num_workers=4, pin_memory=True)

bb = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(
    [{"params": bb, "lr": C["lr_backbone"]},
     {"params": head.parameters(), "lr": C["lr_head"]}],
    weight_decay=C["wd"],
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=C["epochs"])
scaler = torch.amp.GradScaler(DEVICE)


학습 대상 backbone 파라미터: 84.6M / 전체 86.7M


## 학습 루프

In [8]:
best = -1.0
for ep in range(1, C["epochs"] + 1):
    model.train(); head.train()
    losses = []
    for x, y in tr_ld:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        with torch.amp.autocast(DEVICE):
            loss = F.cross_entropy(head(model(x), y), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
    sched.step()

    emb, lab = extract(model, va_ld)
    mAP, t1, t5 = evaluate_map_split(emb, lab, seed=C["seed"])
    print(f"[{ep:02d}/{C['epochs']}] loss {np.mean(losses):.3f} | "
          f"val mAP {mAP:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")

    if mAP > best:
        best = mAP
        torch.save({"model": C["model"], "backbone": model.state_dict(),
                    "id2lab": id2lab, "cfg": cfg, "val_mAP": best}, C["out"])
        print(f"    -> saved {C['out']}  (mAP {best:.4f})")

print(f"\nbest val mAP {best:.4f}")


c:\hyeonkyu\for_mate\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: DataLoader worker (pid(s) 10152, 4348, 9532, 11744) exited unexpectedly

## 최종 평가 — 보호소 데이터 기준

파인튜닝 val 숫자가 아니라, **저장된 backbone 을 원래 `processed_animals` 전체에 돌려 5시드 평균**을 낸다.
(MPDD 로 학습했으면 특히 필수. MPDD 는 MegaDescriptor 학습셋에 포함됐을 수 있어 자기 평가가 부풀려짐)

In [ ]:
ckpt = torch.load(C["out"], map_location=DEVICE)
model.load_state_dict(ckpt["backbone"]); model.eval()

df_all = df_from_folder("processed_animals")
df_all["label"] = df_all["identity"].astype("category").cat.codes
all_ld = DataLoader(ReIDDataset(df_all, "processed_animals", eval_tf),
                    batch_size=64, num_workers=4, pin_memory=True)

emb, lab = extract(model, all_ld)
(mean, std) = evaluate_map_split_mean(emb, lab, seeds=range(5))
print(f"[파인튜닝 backbone / processed_animals]  개체 {df_all['identity'].nunique()} / 이미지 {len(df_all)}")
print(f"  split  mAP {mean[0]:.4f}±{std[0]:.4f} | Top-1 {mean[1]:.4f}±{std[1]:.4f} | Top-5 {mean[2]:.4f}±{std[2]:.4f}")
print("  참고 zero-shot(L-384): mAP ? / Top-1 ~0.53  <- model_test.ipynb 에서 먼저 측정해 둘 것")
